Second version with changes to work with 014_run_eval_code_grader.ipynb

In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
import os
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL", "")

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  \{
    "task": "Description of task",
    "format": "Python/JSON/Regex",
  \},
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.

To properly parse the JSON response, we'll use prefilling and stop sequences
"""

    messages = []

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

<>:8: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:8: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
/tmp/ipykernel_3444076/3372495313.py:8: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  \{


In [4]:
# Testing the Dataset Generation

dataset = generate_dataset()
print(dataset)

# Save to a file
with open('011_dataset_2.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': "Extract the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/path/to/object'", 'format': 'Regex'}, {'task': 'Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message', 'format': 'Python'}, {'task': 'Create a JSON policy document that allows read-only access to a specific S3 bucket', 'format': 'JSON'}]
